# 📊 Notebook 02 — Análisis Descriptivo

**Sistema de Denuncias Ambientales · Uruguay · D Empathy Project**

> *¿Qué está pasando? · ¿Cuánto? · ¿Dónde? · ¿Cuándo?*

Este notebook corresponde a la **Sección 1** del tablero. Acá no hay modelos ni inferencia: solo descripción honesta de los datos. Cuatro miradas:

1. **Temporal** — evolución anual, tendencia mensual con media móvil, heatmap hora × día de semana
2. **Categorías** — treemap, sunburst jerárquico, ranking de subcategorías, área apilada por categoría
3. **Geográfico** — mapa de burbujas por departamento, mapa de puntos, composición por categoría
4. **Perfil del denunciante** — tipo (donut), urgencia (gauge), recurrencia, antecedentes

> ⚠️ **Importante**: los campos `tipo_denunciante`, `urgencia`, `recurrencia` y `denuncia_previa` provienen del formulario ciudadano nuevo. En el histórico están como `NaN`. Las visualizaciones que dependen de estos campos mostrarán mensaje informativo si no hay datos suficientes.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

DATA_DIR = Path("../data")
df = pd.read_parquet(DATA_DIR / "denuncias.parquet")
print(f"Cargado: {df.shape[0]:,} denuncias × {df.shape[1]} columnas")
print(f"Período: {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")

# Catálogo de colores por categoría (consistente con el tablero)
CAT_COLORS = {
    "Fauna silvestre": "#2d6a4f", "Costa y faja costera": "#1b4f72",
    "Contaminación del aire": "#5d6d7e", "Contaminación del agua": "#2e86c1",
    "Residuos y basura": "#e67e22", "Flora y vegetación": "#27ae60",
    "Contaminación sonora": "#f1c40f", "Extracción y act. productivas": "#8b6914",
    "Otro problema ambiental": "#7f8c8d",
}

## KPIs ejecutivos

Cuatro números que un decisor querría tener arriba de su escritorio.

In [ ]:
total = len(df)
n_years = df["año"].nunique()
cat_top = df["categoria_label"].value_counts().index[0]
dept_top = df["departamento"].value_counts().index[0]

print(f"📌 Total de denuncias:        {total:,}")
print(f"📌 Años cubiertos:            {n_years}")
print(f"📌 Promedio por año:          {total/n_years:.0f}")
print(f"📌 Categoría más frecuente:   {cat_top}")
print(f"📌 Departamento líder:        {dept_top}")

# 1️⃣ Análisis temporal

## 1.1 Evolución anual de denuncias

Barras por año con línea horizontal de promedio histórico. Permite ver años atípicos.

In [ ]:
by_year = df.groupby("año").size().reset_index(name="total")
avg = by_year["total"].mean()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=by_year["año"], y=by_year["total"],
    marker_color="#1abc9c",
    hovertemplate="<b>%{x}</b><br>%{y} denuncias<extra></extra>",
))
fig.add_hline(y=avg, line_dash="dot", line_color="#f0883e",
              annotation_text=f"Promedio: {avg:.0f}",
              annotation_font_color="#f0883e")
fig.update_layout(
    title="Denuncias ambientales por año — Uruguay",
    xaxis_title="Año", yaxis_title="Denuncias", height=400,
    plot_bgcolor="white",
)
fig.show()

# Lectura rápida
max_yr = by_year.loc[by_year['total'].idxmax(), 'año']
min_yr = by_year.loc[by_year['total'].idxmin(), 'año']
print(f"\n🔎 El año con MÁS denuncias fue {max_yr} ({by_year['total'].max():,})")
print(f"🔎 El año con MENOS denuncias fue {min_yr} ({by_year['total'].min():,})")

## 1.2 Tendencia mensual + media móvil

La serie cruda por mes es muy ruidosa. Suavizamos con una media móvil de 3 meses para ver la señal.

In [ ]:
monthly = df.groupby(df["timestamp"].dt.to_period("M")).size().reset_index(name="total")
monthly["fecha"] = monthly["timestamp"].dt.to_timestamp()
monthly["ma3"] = monthly["total"].rolling(3, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=monthly["fecha"], y=monthly["total"],
                         mode="lines", name="Mensual",
                         line={"color": "#58a6ff", "width": 1.2}, opacity=0.5))
fig.add_trace(go.Scatter(x=monthly["fecha"], y=monthly["ma3"],
                         mode="lines", name="Media móvil 3 meses",
                         line={"color": "#1abc9c", "width": 2.5}))
fig.update_layout(title="Tendencia mensual con suavizado",
                  xaxis_title="", yaxis_title="Denuncias / mes",
                  height=400, plot_bgcolor="white")
fig.show()

## 1.3 Heatmap hora × día de semana

¿En qué momentos del día y de la semana se registran más denuncias? **Ojo**: esto refleja **cuándo se reporta**, no cuándo ocurre el problema (que sería un análisis distinto a futuro).

In [ ]:
dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
pivot = df.groupby(["dia_semana", "hora"]).size().unstack(fill_value=0)
pivot = pivot.reindex(range(7), fill_value=0)
pivot.index = dias

fig = go.Figure(go.Heatmap(
    z=pivot.values, x=[f"{h:02d}h" for h in range(24)], y=dias,
    colorscale="Tealgrn",
    hovertemplate="<b>%{y} %{x}</b><br>%{z} denuncias<extra></extra>",
))
fig.update_layout(title="Distribución de denuncias por hora y día de semana",
                  xaxis_title="Hora", yaxis_title="", height=320,
                  plot_bgcolor="white")
fig.show()

# Nota: como muchas denuncias vienen con hora=00 (sólo fecha registrada),
# la columna de medianoche está sobre-representada. Es un sesgo de captura
# que se corregirá cuando el formulario nuevo registre hora real.
print(f"\n🔎 % de denuncias con hora=00 (sólo fecha): "
      f"{(df['hora']==0).mean()*100:.1f}%")

# 2️⃣ Análisis de categorías

## 2.1 Treemap — distribución por categoría y subcategoría

El treemap es preferible al pie chart cuando hay muchas categorías. Cada rectángulo es proporcional al volumen.

In [ ]:
agg = df.groupby(["categoria_label", "subcategoria"]).size().reset_index(name="total")
fig = px.treemap(agg, path=["categoria_label", "subcategoria"], values="total",
                 color="total", color_continuous_scale="Teal",
                 title="Distribución de denuncias por categoría → subcategoría")
fig.update_layout(height=500)
fig.show()

## 2.2 Sunburst jerárquico

Misma información que el treemap pero en radial. Lo dejo porque visualmente comunica mejor la idea de "familia" categoría → subcategoría.

In [ ]:
fig = px.sunburst(agg, path=["categoria_label", "subcategoria"], values="total",
                  color="categoria_label",
                  color_discrete_map=CAT_COLORS,
                  title="Jerarquía radial: categoría → subcategoría")
fig.update_layout(height=500)
fig.show()

## 2.3 Ranking de subcategorías (top 15)

In [ ]:
top = df.groupby("subcategoria").size().reset_index(name="total").nlargest(15, "total")
fig = go.Figure(go.Bar(
    x=top["total"], y=top["subcategoria"], orientation="h",
    marker_color="#1abc9c",
    hovertemplate="<b>%{y}</b><br>%{x} denuncias<extra></extra>",
))
fig.update_layout(title="Top 15 subcategorías más denunciadas",
                  yaxis={"categoryorder": "total ascending"},
                  xaxis_title="Denuncias", height=500, plot_bgcolor="white")
fig.show()

## 2.4 Evolución mensual por categoría (área apilada)

Permite ver cómo cambia la **composición** de las denuncias a lo largo del tiempo, no solo el volumen total.

In [ ]:
df_area = df.copy()
df_area["periodo"] = df_area["timestamp"].dt.to_period("M").dt.to_timestamp()
agg_area = (df_area.groupby(["periodo", "categoria_label"])
            .size().reset_index(name="total"))

# Mostramos sólo las 6 categorías top para no saturar
cats_top = df_area["categoria_label"].value_counts().head(6).index.tolist()
agg_area = agg_area[agg_area["categoria_label"].isin(cats_top)]

fig = go.Figure()
for cat in cats_top:
    sub = agg_area[agg_area["categoria_label"] == cat]
    fig.add_trace(go.Scatter(
        x=sub["periodo"], y=sub["total"], mode="lines", name=cat,
        line={"color": CAT_COLORS.get(cat, "#1abc9c"), "width": 1.5},
        stackgroup="one", fill="tonexty"))
fig.update_layout(title="Composición mensual por categoría (top 6)",
                  xaxis_title="", yaxis_title="Denuncias",
                  height=420, plot_bgcolor="white")
fig.show()

# 3️⃣ Análisis geográfico

## 3.1 Mapa de burbujas por departamento

Tamaño y color proporcionales al volumen de denuncias. Vista clara de la distribución territorial.

In [ ]:
DEPT_COORDS = {
    "Artigas": (-30.40, -56.47), "Canelones": (-34.52, -56.28),
    "Cerro Largo": (-32.37, -54.18), "Colonia": (-34.46, -57.84),
    "Durazno": (-33.38, -56.52), "Flores": (-33.55, -56.90),
    "Florida": (-34.10, -56.21), "Lavalleja": (-34.38, -55.24),
    "Maldonado": (-34.91, -54.96), "Montevideo": (-34.90, -56.16),
    "Paysandú": (-32.32, -58.08), "Río Negro": (-33.13, -58.31),
    "Rivera": (-30.91, -55.55), "Rocha": (-34.48, -54.34),
    "Salto": (-31.39, -57.97), "San José": (-34.34, -56.71),
    "Soriano": (-33.45, -58.04), "Tacuarembó": (-31.71, -55.99),
    "Treinta y Tres": (-33.23, -54.38),
}

dept_counts = df.groupby("departamento").size().reset_index(name="total")
dept_counts["lat"] = dept_counts["departamento"].map(lambda d: DEPT_COORDS[d][0])
dept_counts["lon"] = dept_counts["departamento"].map(lambda d: DEPT_COORDS[d][1])

fig = go.Figure(go.Scattermapbox(
    lat=dept_counts["lat"], lon=dept_counts["lon"], mode="markers+text",
    marker={"size": (dept_counts["total"]/dept_counts["total"].max()*55+15).clip(15,70),
            "color": dept_counts["total"], "colorscale": "Teal",
            "opacity": 0.85, "showscale": True,
            "colorbar": {"title": "Denuncias"}},
    text=dept_counts["departamento"], textposition="top center",
    hovertemplate="<b>%{text}</b><br>%{marker.color:.0f} denuncias<extra></extra>",
))
fig.update_layout(
    mapbox={"style": "carto-positron", "center": {"lat":-32.7, "lon":-56.5}, "zoom": 5.3},
    height=500, margin={"t":40,"r":0,"b":0,"l":0},
    title="Volumen de denuncias por departamento")
fig.show()

## 3.2 Mapa de puntos

Muestra puntos individuales (con jitter desde centroide) coloreados por categoría. **Recordatorio**: en datos históricos las coordenadas son *aproximadas a nivel departamento*.

In [ ]:
# Tomo muestra de 500 puntos para que el mapa no se ponga pesado
sample = df.dropna(subset=["latitud","longitud"]).sample(min(500, len(df)), random_state=42)

fig = go.Figure(go.Scattermapbox(
    lat=sample["latitud"], lon=sample["longitud"], mode="markers",
    marker={"size": 7,
            "color": sample["categoria_label"].map(CAT_COLORS).fillna("#1abc9c"),
            "opacity": 0.7},
    hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<extra></extra>",
    customdata=sample[["categoria_label", "departamento"]].values,
))
fig.update_layout(
    mapbox={"style":"carto-positron","center":{"lat":-32.7,"lon":-56.5},"zoom":5.3},
    title="Puntos de denuncia (muestra n=500, jitter sobre centroide departamental)",
    height=500, margin={"t":40,"r":0,"b":0,"l":0})
fig.show()

## 3.3 Composición por categoría en los 8 departamentos top

Barras apiladas. Permite comparar el *perfil ambiental* de cada departamento.

In [ ]:
top_depts = df["departamento"].value_counts().head(8).index.tolist()
agg = (df[df["departamento"].isin(top_depts)]
       .groupby(["departamento", "categoria_label"]).size().reset_index(name="total"))

fig = go.Figure()
for cat in agg["categoria_label"].unique():
    sub = agg[agg["categoria_label"] == cat]
    fig.add_trace(go.Bar(name=cat, x=sub["departamento"], y=sub["total"],
                         marker_color=CAT_COLORS.get(cat, "#1abc9c")))
fig.update_layout(barmode="stack", title="Composición de categorías por departamento (top 8)",
                  yaxis_title="Denuncias", height=450, plot_bgcolor="white",
                  legend={"font":{"size": 9}, "orientation": "h", "y": -0.3})
fig.show()

# 4️⃣ Perfil del denunciante

> ⚠️ Esta sección depende de campos del formulario ciudadano (`tipo_denunciante`, `urgencia`, `recurrencia`, `denuncia_previa`). En el dataset histórico estos campos son `NaN`. Cuando llegue suficiente data del formulario los gráficos se llenan automáticamente.

In [ ]:
# % de cobertura de los campos del formulario
cols_form = ["tipo_denunciante", "urgencia", "recurrencia", "denuncia_previa"]
cobertura = (df[cols_form].notna().sum() / len(df) * 100).round(2)
print("Cobertura de campos del formulario (% no nulos):")
print(cobertura.to_string())

In [ ]:
if df["tipo_denunciante"].notna().any():
    counts = df["tipo_denunciante"].value_counts().reset_index()
    counts.columns = ["tipo", "total"]
    fig = go.Figure(go.Pie(labels=counts["tipo"], values=counts["total"], hole=0.55,
                            marker_colors=["#1abc9c", "#58a6ff", "#8b949e"]))
    fig.update_layout(title="Tipo de denunciante", height=350)
    fig.show()
else:
    print("⚠️ Sin datos de tipo_denunciante todavía.\n"
          "   Este gráfico se llenará cuando lleguen denuncias por el formulario.")

---
## ✅ Cierre

Lo que vimos en este notebook:

- **Temporal**: hay una clara concentración en 2017-2018 (sobre todo 2017 con ~1.000 denuncias) y luego una vuelta al volumen "normal" en 2023. La interpretación de esos picos amerita más investigación (¿campaña pública? ¿cambio de catalogación interna del organismo?).
- **Categorías**: "Otro problema ambiental" lidera (826) — es una señal de que el catálogo tiene un cajón muy genérico que conviene afinar. Le siguen "Aire" (755), "Fauna" (660), "Agua" (639) y "Residuos" (570).
- **Geografía**: Canelones, Montevideo y Maldonado concentran el grueso (es esperable por densidad poblacional y costa). La distribución de tipos de denuncia varía mucho entre departamentos (Maldonado pesa mucho en costa, Montevideo en aire/ruido).
- **Perfil**: pendiente de datos del formulario.

**Próximo notebook**: `03_diagnostico.ipynb` — buscar patrones latentes: correlaciones, anomalías, scoring de riesgo.